In [ ]:
!apt-get update -q
!apt-get install -y chromium-browser chromium-driver -q
!pip install selenium beautifulsoup4 textblob wordcloud webdriver_manager --quiet
!python -m textblob.download_corpora --quiet

In [ ]:
!pip install google-colab-selenium --quiet

from google_colab_selenium import ChromeDriver

driver = ChromeDriver()
driver.get(URL)

In [ ]:
URL          = "https://www.youtube.com/watch?v=OTlVmfO_j0Q"
MAX_COMMENTS = 40
SCROLL_PAUSE = 2.5

import time, re, json
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from transformers import pipeline
from IPython.display import display, HTML
import pandas as pd

STAR_LABELS = {
    1: ("⭐",        "Muy Negativo / Odio"),
    2: ("⭐⭐",      "Negativo"),
    3: ("⭐⭐⭐",    "Neutral / Ambivalente"),
    4: ("⭐⭐⭐⭐",  "Positivo"),
    5: ("⭐⭐⭐⭐⭐","Muy Positivo / Excelente"),
}

In [ ]:
from google_colab_selenium import ChromeDriver

def get_driver():
    """Configura Chrome para Colab usando la librería especializada."""
    driver = ChromeDriver()
    return driver

def scrape_comments(url: str, max_comments: int) -> list:
    print("🚀 Abriendo Chrome headless...")
    driver = get_driver()
    driver.get(url)
    time.sleep(4)

    print("📜 Scrolleando para cargar comentarios...")
    comments_found = []

    for i in range(20):
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(SCROLL_PAUSE)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        comments_found = soup.find_all("yt-attributed-string", id="content-text")

        print(f"   Scroll {i+1:02d} → {len(comments_found)} comentarios encontrados")
        if len(comments_found) >= max_comments:
            break

    driver.quit()

    seen, clean = set(), []
    for tag in comments_found[:max_comments]:
        text = tag.get_text(separator=" ", strip=True)
        if text and len(text) > 3 and text not in seen:
            seen.add(text)
            clean.append(text)

    print(f"\n✅ {len(clean)} comentarios únicos extraídos")
    return clean


comments = scrape_comments(URL, MAX_COMMENTS)

In [ ]:
sentiment = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True,
    max_length=512,
)

In [ ]:
print(" Analizando comentarios...\n")
results = []

for i, comment in enumerate(comments, 1):
    pred   = sentiment(comment)[0]
    stars  = int(re.search(r"(\d)", pred["label"]).group(1))
    score  = round(pred["score"] * 100, 2)
    emoji, label = STAR_LABELS[stars]

    results.append({
        "#":             i,
        "Comentario":    comment,
        "Estrellas":     stars,
        "Sentimiento":   f"{emoji} {label}",
        "Score (%)": score,
    })

    print(f"[{i:02d}] {comment[:65]}{'...' if len(comment)>65 else ''}")
    print(f"      → {emoji} {label}  |  Confianza: {score}%\n")

df = pd.DataFrame(results)
print("Análisis completado")

In [ ]:
dist = df["Estrellas"].value_counts().sort_index()
dist = dist.reindex(range(1, 6), fill_value=0)


import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
colores = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']
barras = plt.bar(dist.index, dist.values, color=colores)

etiquetas = [f"{i} estrella" if i == 1 else f"{i} estrellas" for i in dist.index]
plt.xticks(dist.index, etiquetas)

plt.title("Distribución de Calificaciones (Estrellas)", fontsize=14)
plt.xlabel("Nivel de sentimiento", fontsize=12)
plt.ylabel("Número de comentarios", fontsize=12)

max_y = max(dist.values) if max(dist.values) > 0 else 1
plt.ylim(0, max_y * 1.1)

for i, v in enumerate(dist.values):
    if v > 0:
        plt.text(dist.index[i], v + (max_y * 0.03), str(v), ha='center', va='bottom', fontsize=10)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
dist = df["Estrellas"].value_counts().sort_index()
dist = dist.reindex(range(1, 6), fill_value=0)

total = len(df)
avg = df["Estrellas"].mean()

print("\n  Distribución por estrellas:")
for s in range(1, 6):
    n = dist.loc[s]
    print(f"  {s} estrella{'s' if s>1 else ''} : {n}")

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
colores = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']
plt.bar(dist.index, dist.values, color=colores)

etiquetas = [f"{i} estrella" if i == 1 else f"{i} estrellas" for i in dist.index]
plt.xticks(dist.index, etiquetas)

plt.title("Distribución de Calificaciones (Estrellas)", fontsize=14)
plt.xlabel("Nivel de sentimiento", fontsize=12)
plt.ylabel("Número de comentarios", fontsize=12)

max_y = max(dist.values) if max(dist.values) > 0 else 1
plt.ylim(0, max_y * 1.1)

for i, v in enumerate(dist.values):
    if v > 0:
        plt.text(dist.index[i], v + (max_y * 0.03), str(v), ha='center', va='bottom', fontsize=10)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()